In [1]:
import load_assess_image as load_assess
import numpy as np
import pandas as pd
import os
import skimage
from skimage.measure import regionprops_table
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor 

In [2]:
main_dir = rf'D:\thu\HTAN\images\NST'
mask_dir = rf'D:\thu\HTAN\images\mask\NST'
save_dir = rf'D:\thu\HTAN\images\quantification\NST'
os.makedirs(save_dir, exist_ok=True)

In [3]:
def single_sample_quantification(file_name):
    stem = file_name.replace('.npz', '')
    output_path = os.path.join(mask_dir, file_name)
    data = np.load(output_path)
    nuclei_mask = data['nuclei']
    cells_mask  = data['cells']

    image_test=load_assess.AssessImage(main_dir,stem+'.ome.tif')
    shape_props = regionprops_table(
        nuclei_mask,
        properties=['label', 'area', 'centroid', 'equivalent_diameter_area',
                    'solidity', 'eccentricity', 'axis_major_length', 'axis_minor_length', 'perimeter']
    )
    df = pd.DataFrame(shape_props)
    df['sample_id'] = stem

    for biomarker, channel_index in image_test.biomarker_channel.items():
        image_biomarker = np.array(image_test.assess_single_image(biomarker=biomarker, resolution_level=0))
        intensity_props = regionprops_table(
            nuclei_mask,
            intensity_image=image_biomarker,
            properties=['label', 'intensity_mean', 'intensity_median', 'intensity_std']
        )
        intensity_df = pd.DataFrame(intensity_props).rename(columns={
            'intensity_mean':   f'{biomarker}_mean',
            'intensity_median': f'{biomarker}_median',
            'intensity_std':    f'{biomarker}_std',
        })

        df = df.merge(intensity_df, on='label')

    output_path = os.path.join(save_dir, stem + '.csv') 
    df.to_csv(output_path, index=True)
    print(f"Saved {len(df)} cells → {output_path}") 

In [ ]:
with ProcessPoolExecutor (max_workers=26) as executor:
    npz_files = [f for f in os.listdir(mask_dir) if f.endswith('.npz')]
    futures = {executor.submit(single_sample_quantification, f): f for f in npz_files}
    for future in as_completed(futures):
        f = futures[future]
        try:
            future.result()
        except Exception as e:
            print(f"{f} failed: {e}")


Traceback (most recent call last):
  File "d:\conda_envs\instanseg\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "d:\conda_envs\instanseg\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "d:\conda_envs\instanseg\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is closed
Traceback (most recent call last):
  File "d:\conda_envs\instanseg\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "d:\conda_envs\instanseg\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "d:\conda_envs\instanseg\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is closed
Traceback (most recent call last):
  File "d:\conda_envs\instanseg\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "d:\conda_envs\instanseg\Li

NameError: name 'as_completed' is not defined

: 